In [ ]:
# 1. Imports, utilities, and config (run from the repository root)
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd()
MODEL_ROOT = PROJECT_ROOT / 'logs' / 'ByModel'
OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'classical_training_dynamics.png'
module_path = PROJECT_ROOT / 'src' / 'analytics' / 'training' / 'classical_training_dynamics.py'
spec = importlib.util.spec_from_file_location('classical_training_dynamics', module_path)
dynamics = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dynamics
spec.loader.exec_module(dynamics)
ARCHITECTURES = dynamics.ARCHITECTURES
discover_model_configs = dynamics.discover_model_configs
prepare_curves = dynamics.prepare_curves
save_figure = dynamics.save_figure


In [ ]:
# 2. Crawl and find .model-anchored classical configurations
CLASSICAL_ROOT = MODEL_ROOT / 'Classical'
candidates = discover_model_configs(CLASSICAL_ROOT)
for model_dir in candidates:
    print(model_dir.relative_to(MODEL_ROOT), '=>', (model_dir / '.model').read_text(encoding='utf-8').strip())


In [ ]:
# 3. Manually select variants, then load and validate seed 0 and 42
selected = {
    'MLP': CLASSICAL_ROOT / 'mlp' / 'hidden_sizes=[128,64]',
    'ConvNet': CLASSICAL_ROOT / 'convnet' / 'conv_channels=[32,64],fc_sizes=[128]',
    'ResNet18': CLASSICAL_ROOT / 'resnet18',
    'MLP-Mixer': CLASSICAL_ROOT / 'mlpmixer',
}
curves = prepare_curves(selected)
for name in ARCHITECTURES:
    curve = curves[name]
    print(f'{name}: {len(curve.epochs)} plotted of {curve.available_epochs} recorded epochs; source files: {curve.source_files}; pretraining validation removed: {curve.skipped_pretraining_validation}')


In [ ]:
# 4. Generate the four-panel PNG from unsmoothed epoch-level means
figure_path = save_figure(curves, OUTPUT_PATH)
print(figure_path)
